# Class 1 & 2: NLP and Search
## Learning Notebook Part 2 - Advanced Search & Clustering

**Welcome to Part 2!** In this notebook, we'll build on the foundation from Part 1:

**You should have completed Part 1** where you learned:
- Keyword search (simple and multiple keyword search)
- Simple tokenization
- Text preprocessing (cleaning, tokenization, regex)
- Bag of Words (word counts - converting text to numbers)
- Understanding vector representations

**Now in Part 2, you'll learn:**
- 📊 **TF-IDF**: Improved text representation (concept - you'll implement in exercises!)
- 🎯 **Similarity-Based Search**: Finding relevant documents using TF-IDF + cosine similarity
- 📦 **Clustering**: Automatically organizing documents with K-Means

Let's dive in!


## Setup


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# For better output display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


### Load the Data


In [ ]:
# Load movie descriptions
# If running in Google Colab and data file doesn't exist, download it from GitHub
import os

if not os.path.exists('data/movies.csv'):
    print("Data file not found. Downloading from GitHub...")
    os.makedirs('data', exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/samsung-ai-course/8th-9th-edition/main/Chapter%202%20-%20Natural%20Language%20Processing/Class%201%20%26%202%20-%20NLP%20and%20Search/data/movies.csv'
    urllib.request.urlretrieve(url, 'data/movies.csv')
    print("✓ Data file downloaded successfully!")

df = pd.read_csv('data/movies.csv')
print(f"Loaded {len(df)} movies")
df.head()

## Sparse vs Dense Vectors (Quick Reminder)

Before we dive into TF-IDF, let's quickly visualize sparse vs dense vectors:


In [ ]:
# Example: Sparse vector (TF-IDF will create this)
# Let's use a small example first
sample_docs = df['description'].head(3).tolist()

# Create TF-IDF vectors (sparse!)
vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sample_docs)

print("Sparse TF-IDF Matrix Shape:", tfidf_matrix.shape)
print(f"Total elements: {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]}")
print(f"Non-zero elements: {tfidf_matrix.nnz}")
print(f"Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print("\nFirst document vector (first 20 values):")
print(tfidf_matrix[0].toarray()[0][:20])

# Compare to dense vector (random example)
dense_example = np.random.rand(50)
print(f"\n\nDense vector (50 dimensions, all non-zero):")
print(dense_example[:20])


**Key Takeaway**: 
- **Sparse vectors** (BoW, TF-IDF) are interpretable - we know what each dimension means
  - **BoW** (from Part 1): Word counts (e.g., "python" appears 3 times)
  - **TF-IDF** (this part): Word frequencies weighted by importance
- **Dense vectors** (embeddings) are more powerful - they capture meaning and relationships
- For now, we'll use **TF-IDF** (sparse) as it's simple and interpretable. Next class, we'll see dense embeddings!


## TF-IDF: Term Frequency-Inverse Document Frequency

**TF-IDF** is an improvement over simple Bag of Words. It weighs words by:
- **TF (Term Frequency)**: How often a word appears in a document (higher = more important to that document)
- **IDF (Inverse Document Frequency)**: How rare a word is across all documents (rare words are more informative)

**Formula**: TF-IDF = TF × IDF

**Why it works**: Common words like "the", "a" have high TF but low IDF (they appear in many documents), so their TF-IDF is low. Important words like "Python", "space" have high TF-IDF.

**Important**: You'll implement TF-IDF from scratch in the **Exercise Notebook**! Here we'll just create TF-IDF vectors together (using scikit-learn) so we can use them for similarity search and clustering. The exercise will teach you how TF-IDF actually works step-by-step.


In [ ]:
# TF-IDF Vectors - Let's create them together using scikit-learn!
# NOTE: You'll implement TF-IDF from scratch in Exercise 3!
# Here we'll use scikit-learn to create TF-IDF vectors for similarity search and clustering

# TODO (Together): Create TF-IDF vectorizer
# What parameters should we use?
vectorizer = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)  # TODO: Discuss these parameters

# TODO (Together): Fit and transform all documents
# What does 'fit' do? What does 'transform' do?
tfidf_vectors = vectorizer.fit_transform(df['description'])  # TODO: Complete this

print(f"TF-IDF Matrix Shape: {tfidf_vectors.shape}")
print(f"(Number of documents, Vocabulary size)")
print(f"\nVocabulary (first 20 words):")
print(vectorizer.get_feature_names_out()[:20])

print(f"\n💡 Remember: You'll learn to implement TF-IDF step-by-step in Exercise 3!")
print(f"   For now, we're using scikit-learn to create vectors for similarity search and clustering.")


## Similarity-Based Search with TF-IDF

**Now let's implement similarity-based search together!** This finds relevant documents based on similarity, not just exact keyword matches.

The idea:
1. Convert query to TF-IDF vector (using the same vectorizer)
2. Compare with all document vectors using **Cosine Similarity**
3. Return most similar documents (highest similarity scores)

**Cosine Similarity**: Measures the angle between two vectors. Range: -1 to 1
- **1** = identical direction (very similar)
- **0** = perpendicular (no similarity)
- **-1** = opposite direction (very different)

**Why cosine?** It measures similarity regardless of document length!


In [ ]:
# Similarity-Based Search - Let's implement this together!
def search_tfidf(query, vectorizer, tfidf_vectors, df, top_k=5):
    """
    Similarity-based search using TF-IDF and cosine similarity
    
    Steps:
    1. Convert query to TF-IDF vector
    2. Calculate cosine similarity with all documents
    3. Get top_k most similar documents
    4. Return results as DataFrame
    """
    # TODO (Together): Step 1 - Convert query to TF-IDF vector using the same vectorizer
    # What method should we use? transform() or fit_transform()?
    query_vector = vectorizer.transform([query])  # TODO: Complete this
    
    # TODO (Together): Step 2 - Calculate cosine similarity with all document vectors
    # How do we compare query_vector with tfidf_vectors?
    similarities = cosine_similarity(query_vector, tfidf_vectors)[0]  # TODO: Complete this
    
    # TODO (Together): Step 3 - Get indices of top_k most similar documents
    # How do we find the indices of the highest similarity scores?
    # Hint: Use argsort() and reverse order
    top_indices = similarities.argsort()[-top_k:][::-1]  # TODO: Complete this
    
    # TODO (Together): Step 4 - Build results DataFrame
    results = []
    for idx in top_indices:
        results.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'similarity': similarities[idx],
            'description': df.iloc[idx]['description']
        })
    
    return pd.DataFrame(results)

# Let's test it!
example_query = "space exploration adventure"
print(f"Query: '{example_query}'")
print("\nResults:")
results = search_tfidf(example_query, vectorizer, tfidf_vectors, df, top_k=5)
print(results[['title', 'similarity']])


### Compare: Keyword Search vs Similarity-Based Search (TF-IDF)

Let's see how they differ:

**Note**: TF-IDF similarity search is better than simple keyword matching, but it's still fundamentally keyword-based (not true semantic search). True semantic search that understands synonyms and meaning requires embeddings (Class 3)!


In [ ]:
# Comparison: Keyword Search vs Similarity-Based Search
# You'll implement both in exercises!

print("=" * 60)
print("KEY DIFFERENCES: Keyword Search vs Similarity Search")
print("=" * 60)

print("\nKeyword Search (from Part 1 - Exercise 4):")
print("  ✅ Fast and simple")
print("  ✅ Finds exact word matches")
print("  ✅ Multiple keyword search allows more specific queries")
print("  ❌ No ranking - all matches are equal")
print("  ❌ 'mind-bending' won't find 'psychological thriller'")
print("  ❌ Limited to exact words")

print("\nSimilarity-Based Search (TF-IDF - Exercise 5):")
print("  ✅ Ranks by importance (TF-IDF weighted)")
print("  ✅ Better than keyword search")
print("  ✅ Finds documents with similar word patterns")
print("  ⚠️ Still keyword-based (not true semantic)")
print("  ⚠️ 'mind-bending' might match 'psychological' IF they share other words")
print("  ❌ Still cannot understand synonyms or true meaning")

print("\nTrue Semantic Search (Embeddings - Class 3!):")
print("  ✅ Understands meaning and synonyms")
print("  ✅ 'space' and 'cosmic' are similar (semantic similarity)")
print("  ✅ True understanding of meaning")
print("  ⚠️ Requires embeddings (Class 3)")

print("\n💡 Key Point: TF-IDF is BETTER than keyword search,")
print("   but it's still SYNTAX-based (word patterns), not SEMANTIC (meaning).")
print("   Semantic = meaning. True semantic search requires embeddings!")

print("\n💡 Implementation:")
print("   - Exercise 4: Keyword search")
print("   - Exercise 5: Similarity-based search with TF-IDF")
print("   - Class 3: Semantic search with embeddings")


## Clustering Documents

Now let's group similar movies together using **K-Means Clustering** (unsupervised learning!).

**Goal**: Automatically discover groups of similar documents without labels.


In [ ]:
# Clustering - Let's cluster movies together!
n_clusters = 4  # We know there are roughly 4-5 genres

# TODO (Together): Create KMeans clusterer
# What parameters should we use? (n_clusters, random_state, n_init)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)  # TODO: Complete this

# TODO (Together): Fit and predict clusters
# What does fit_predict do? How is it different from fit() then predict()?
clusters = kmeans.fit_predict(tfidf_vectors)  # TODO: Complete this

# Add cluster labels to dataframe
df['cluster'] = clusters

# Display clusters
print("Movie Clusters:")
print("=" * 60)
for cluster_id in range(n_clusters):
    cluster_movies = df[df['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id} ({len(cluster_movies)} movies):")
    print("-" * 40)
    for idx, row in cluster_movies.head(5).iterrows():  # Show first 5 in each cluster
        print(f"  - {row['title']} ({row['genre']})")


### Visualizing Clusters

Let's reduce the dimensions to 2D using PCA (Principal Component Analysis) so we can visualize the clusters:


In [ ]:
# Visualizing Clusters - Concept
# You can try this in Exercise 6 after implementing clustering!

print("=" * 60)
print("Visualizing Clusters with PCA (Concept):")
print("=" * 60)

print("\nProblem: TF-IDF vectors are high-dimensional (100+ dimensions)")
print("  - Can't visualize in 100D space!")
print("  - Need to reduce to 2D for plotting")

print("\nSolution: Principal Component Analysis (PCA)")
print("  - Reduces dimensions while preserving information")
print("  - Projects high-dimensional vectors to 2D")
print("  - Each point in 2D = one document")

print("\nVisualization Process:")
print("  1. Convert sparse TF-IDF matrix to dense")
print("  2. Apply PCA to reduce to 2 dimensions")
print("  3. Plot documents in 2D space")
print("  4. Color by cluster assignment")
print("  5. Add movie titles as labels")

print("\nWhat to look for:")
print("  ✅ Documents in same cluster should be close together")
print("  ✅ Different clusters should be separated")
print("  ✅ Clusters should make semantic sense (similar genres)")

print("\nVariance Explained:")
print("  - PCA preserves as much information as possible")
print("  - Example: 80% variance explained = 80% of information kept")
print("  - Lower variance = more information lost in 2D projection")

print("\n💡 Implementation: After completing Exercise 6, try visualizing clusters!")
print("   Use PCA to reduce dimensions and matplotlib to plot.")


## Summary

In this notebook, you learned:

1. ✅ **TF-IDF**: Improved text representation that weights words by importance
2. ✅ **Similarity-Based Search**: Using TF-IDF + cosine similarity to find relevant documents
3. ✅ **Clustering**: Grouping similar documents with K-Means (unsupervised learning)

### Next Steps

- **Practice**: Complete the Exercise Notebook to implement TF-IDF from scratch!
- **Next Class**: You'll learn about embeddings for true semantic search

**Note**: Production search systems, evaluation metrics, and connections to supervised learning are covered in the theory slides (Class 1.md) - read those for the complete picture!
